# 02｜OpenAI APIで文章・画像・PDFを使う

APIは、PythonからAIへ質問を送る窓口です。**文章だけの質問 → 画像・PDFを添えた質問**を試します。CPUで実行できます（目安15〜20分）。

必要なものは**教員から個別に受け取ったAPIキー**と、このNotebookだけ。画像・PDFの練習では配布ZIPの`materials/`も使います。`connection.local.yaml`は不要です。

**1 → 2 → 3**で文章の実習。続いて**4 → 5**で画像・PDFの実習です。料金がかかるのは3・5で`SEND = True`にして実行したときだけ。質問・回答・気付いたことは各自のメモ帳へ残してください。

## 何を入力し、何を確かめるか

以下は**質問と確認点の例**です。APIで得た回答や模範解答ではありません。使いたい例を質問欄へ自分で貼り付けます。

| 試すこと | 入力する例 | 回答で確かめること |
|---|---|---|
| 文章（3） | 「APIとは何ですか。学部3年生向けに、身近な例を1つ挙げて3文で説明してください。」 | 説明・具体例があり、3文に収まるか |
| 画像（4→5） | `materials/fig2_full.png`と「図2bの縦軸と横軸は何を表しますか。読めない文字は推測せず教えてください。」 | 原図の軸ラベルと一致するか |
| PDF（4→5） | `materials/Paper2Agent_Nature.pdf`と「図1aにあるMCPサーバーの役割を説明し、根拠となる本文のPDFページを示してください。」 | 図1aと本文を照合できるか |

PDFは`materials/ai_scientist_nature.pdf`でも試せます。資料にない説明や出典は、原文・出版社サイトで確かめます。

## 1. ライブラリを入れる（初回だけ）

`INSTALL`にチェックして▶。終わったらチェックを外します。追加するのはOpenAI公式のPythonライブラリ1つです。

In [ ]:
# @title 1. OpenAIライブラリを準備する
INSTALL = False # @param {type:"boolean"}
if INSTALL:
    import subprocess
    import sys
    from pathlib import Path
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--cache-dir', str(Path.cwd() / 'build/pip-cache'), 'openai==2.54.0',
    ])
    print('導入が完了しました。2へ進みます。')

## 2. モデルとAPIキーを用意する

モデルを選び、`CONNECT`にチェックして▶。**伏字の欄に個別キーを入力**します。キーをコード・質問・メモに書かないでください。接続先はOpenAIの直接APIです。この時点では質問を送りません。

通常は`gpt-6-luna`。モデルを変えるときは、このセルも実行し直します。

In [ ]:
# @title 2. モデルを選び、キーを入力する
CONNECT = False # @param {type:"boolean"}
model = 'gpt-6-luna' # @param ["gpt-6-luna", "gpt-6-sol", "gpt-4.1-mini"]
max_output_tokens = 2048 # @param {type:"integer"}

from getpass import getpass
from openai import OpenAI, APIError

if model not in ('gpt-6-luna', 'gpt-6-sol', 'gpt-4.1-mini'):
    raise ValueError('授業で使う3モデルから選んでください。')
if type(max_output_tokens) is not int or max_output_tokens < 1:
    raise ValueError('出力上限は1以上の整数にしてください。')
settings = {'model': model, 'max_output_tokens': max_output_tokens, 'store': False}
if model == 'gpt-6-luna':
    settings['reasoning'] = {'effort': 'none'}
elif model == 'gpt-6-sol':
    settings['reasoning'] = {'effort': 'low'}

if 'client' not in globals():
    client = None
if CONNECT:
    if client is not None:
        client.close()
    client = None
    api_key = getpass('個別APIキー（表示・保存しません）: ').strip()
    try:
        if not api_key:
            raise ValueError('APIキーが空です。')
        client = OpenAI(api_key=api_key, base_url='https://api.openai.com/v1',
                        max_retries=0, timeout=120.0)
    finally:
        del api_key
    print('準備完了。モデル:', model, '／ 出力上限:', max_output_tokens)

def show_response(response) -> None:
    """APIが返した文章と状態を表示する。回答を補ったり保存したりしない。"""
    print('状態:', response.status, '／ 応答モデル:', response.model)
    print(response.output_text or '文章の回答はありません。')
    if response.status != 'completed':
        reason = getattr(response.incomplete_details, 'reason', None)
        print('未完了の理由:', reason or '不明')
        if reason == 'max_output_tokens':
            print('2で出力上限を増やすか、質問を短くして再実行してください。')
        else:
            print('完了した回答ではありません。表示を教員・TAに確認してください。')
    if response.usage is not None:
        print('使用トークン:', response.usage.input_tokens, '入力 /', response.usage.output_tokens, '出力')

## 3. 文章で質問する

三重引用符`"""`の間に質問を書きます。`SEND = True`にして▶を押すと、`client.responses.create(...)`が質問を送り、`show_response`が回答を表示します。

**毎回、新しい質問として送ります。前の質問・回答は含めません。** 条件や質問を変えて比べ、分かりやすかった点・確かめたい点をメモしてください。

In [ ]:
question = """
"""
SEND = False  # 質問を確認し、送るときにTrueにする

response = None
if SEND:
    if client is None:
        raise ValueError('先に2でキーを入力してください。')
    if not question.strip():
        raise ValueError('三重引用符の間に質問を書いてください。')
    print('送る質問:', question)
    try:
        response = client.responses.create(input=question, **settings)
    except APIError as error:
        # キー等が含まれる可能性のあるエラー本文は表示しない。
        raise RuntimeError(f'API送信に失敗: {type(error).__name__} / HTTP {getattr(error, "status_code", None)}。下の「困ったとき」を確認してください。') from None
    show_response(response)
else:
    print('質問を記入し、SEND = Trueで実行してください。')

## 4. 画像・PDFを1つ選ぶ

`UPLOAD`にチェックして▶ → 「ファイルを選択」で、配布ZIPの`materials/`にある画像またはPDFを1つ選びます。パスのコピーや入力形式の選択は不要です。

PNG・JPEGは画像のまま、**PDFは全ページ**を5で送ります。ページを絞る場合は、必要なページだけのPDFを先に用意してください。ここではColabへ置くだけで、OpenAIにはまだ送りません。APIへの入力が許可された資料を使ってください。

In [ ]:
# @title 4. 画像・PDFを選ぶ
UPLOAD = False # @param {type:"boolean"}
local_path = ''  # 自分のPCで実行する場合のみ、ここへファイルのパスを記入

import base64
from pathlib import Path
from IPython.display import Image, display

if 'attachment' not in globals():
    attachment = None
if UPLOAD:
    attachment = None  # キャンセル・失敗時に前のファイルを送らない
    if local_path.strip():
        path = Path(local_path)
        filename, data = path.name, path.read_bytes()
    else:
        from google.colab import files
        directory = Path.cwd() / 'private/uploads'
        directory.mkdir(parents=True, exist_ok=True)
        uploaded = files.upload(target_dir=str(directory))
        if len(uploaded) != 1:
            raise ValueError('画像またはPDFを1つだけ選んでください。')
        saved_name, data = next(iter(uploaded.items()))
        filename = Path(saved_name).name
    suffix = Path(filename).suffix.lower()
    formats = {'.png': ('image/png', b'\x89PNG\r\n\x1a\n'),
               '.jpg': ('image/jpeg', b'\xff\xd8\xff'),
               '.jpeg': ('image/jpeg', b'\xff\xd8\xff'),
               '.pdf': ('application/pdf', b'%PDF-')}
    if suffix not in formats:
        raise ValueError('PNG・JPEG・PDFのいずれかを選んでください。')
    mime, signature = formats[suffix]
    if not data.startswith(signature):
        raise ValueError('ファイルの中身と拡張子が一致しません。元ファイルを確認してください。')
    if len(data) >= 50_000_000:
        raise ValueError('50 MB未満のファイルを選んでください。')
    encoded = base64.b64encode(data).decode('ascii')
    data_url = f'data:{mime};base64,{encoded}'
    if suffix == '.pdf':
        prepared = {'type': 'input_file', 'filename': filename, 'file_data': data_url}
        print('PDF全ページを送ります:', filename)
    else:
        prepared = {'type': 'input_image', 'image_url': data_url}
        display(Image(data=data))
        print('この画像を送ります:', filename)
    attachment = prepared
    print('ファイルサイズ:', len(data), 'bytes。5で質問します。')

## 5. ファイルについて質問する

三重引用符の間に質問を書き、`SEND = True`にして▶。**4で選んだファイルと、この質問だけ**を送ります。`input_image`は画像、`input_file`はPDF、`input_text`は質問を表します。

同じファイルへ別の質問をするなら5だけ、別ファイルなら4→5を実行します。原図・本文と回答を比べ、気付いたことをメモしてください。

In [ ]:
question = """
"""
SEND = False  # ファイルと質問を確認し、送るときにTrueにする

response = None
if SEND:
    if client is None:
        raise ValueError('先に2でキーを入力してください。')
    if attachment is None:
        raise ValueError('先に4で画像・PDFを選んでください。')
    if not question.strip():
        raise ValueError('三重引用符の間に質問を書いてください。')
    print('送るファイル:', filename, '／ 質問:', question)
    try:
        response = client.responses.create(
            input=[{'role': 'user', 'content': [
                attachment, {'type': 'input_text', 'text': question},
            ]}],
            **settings,
        )
    except APIError as error:
        raise RuntimeError(f'API送信に失敗: {type(error).__name__} / HTTP {getattr(error, "status_code", None)}。下の「困ったとき」を確認してください。') from None
    show_response(response)
else:
    print('質問を記入し、SEND = Trueで実行してください。')

## 困ったとき・次に試すこと

- **`incomplete`で止まる**：理由が`max_output_tokens`なら、2の上限を4096などに増やして実行するか、質問を絞ります。再送も課金対象です。
- **401／403**：キーとモデルの許可を教員・TAへ確認。**429**：利用上限・混雑を確認。**通信エラー・タイムアウト**：課金されたかは不明です。自動では再送しません。
- **質問を変えたのに前の回答を参照しない**：各回は独立しています。必要な前提や原文を質問へ含めてください。Web検索も自動では行いません。

発展：モデル・質問・添付のどれか1つを変えて比べます。2のコードにある`settings`が送信する設定です。GPT-4.1 miniでは`settings['temperature'] = 0.7`で回答のばらつきを調整できます。旧モデルのPredicted Outputsなど、別のAPI機能は[対応機能のメモ](../docs/capability_audit.md)を参照してください。

キーは他者へ共有しません。利用額の上限は教員がAPI側で管理します。必要な質問・回答・モデル名・入力資料・自分の考察をメモしてから終了してください。モデルの変更・入力変換・再試行を自動で行う処理はありません。

**公式資料**：[Pythonでの基本操作](https://developers.openai.com/api/docs/quickstart)・[画像の入力](https://developers.openai.com/api/docs/guides/images-vision)・[PDFの入力](https://developers.openai.com/api/docs/guides/file-inputs)。Notebookは2026-10-03にこれらと公式SDK 2.54.0を確認して作成。Colab実機と直接APIの試験結果は[検証記録](../docs/verification.md)に分けて記載します。